<a href="https://colab.research.google.com/github/DevisriprasadSoma/MLflyrank/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DevisriprasadSoma/MLflyrank/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice

For Lane 2 — Refresh / Content Opportunity Scoring, I will use **Logistic Regression** as the first ML model.

The task is to estimate the likelihood of content-performance decline using information available at the decision moment. Logistic Regression fits this binary prediction setup and provides an interpretable baseline model that can be compared with the Week-4 rule-based score.

I chose this method because the goal is not to maximize model complexity. The goal is to test whether a simple, interpretable ML model can provide useful decision-support ranking beyond the existing rule-based baseline.

The model uses only the selected development features. Label-derived fields such as `trend_direction` and `trend_pct` are not used as model features.


In [58]:
import os
import pandas as pd
import numpy as np

repo = "/content/flyrank-ml-internship-starter"
raw_path = os.path.join(
    repo,
    "data/raw/content_refresh_anonymized.csv"
)

if not os.path.exists(raw_path):
    !git clone -q https://github.com/flyrank-bih/flyrank-ml-internship-starter.git

df = pd.read_csv(raw_path)

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Shape:", df.shape)

Rows: 30000
Columns: 44
Shape: (30000, 44)


## 2. Split design

I use a **stratified 80/20 holdout split** because the starter content-level dataset does not expose an observation-date field that can support a true chronological split.

The training data contains 24,000 rows and the held-out test data contains 6,000 rows. Stratification keeps the decline rate similar between the two groups.

The target is created from `trend_direction`, while label-derived fields such as `trend_direction` and `trend_pct` are excluded from the model features.

The evaluation is intended as a development-level comparison between the Logistic Regression model and the Week-4 rule-based baseline. It should not be interpreted as a true future-month or production validation.


In [59]:
model_df = df.copy()

# Binary target: decline vs no decline
model_df["target"] = (
    model_df["trend_direction"]
    .astype(str)
    .str.lower()
    .eq("down")
    .astype(int)
)

# Features available from the current observation window
feature_cols = [
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "word_count",
    "char_count",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
]

# Keep model features + target
feature_df = model_df[feature_cols + ["target"]].copy()

# Make numeric values safe
for col in feature_cols:
    feature_df[col] = pd.to_numeric(
        feature_df[col],
        errors="coerce"
    ).fillna(0)

print("Feature dataframe shape:", feature_df.shape)

print("\nTarget distribution:")
print(feature_df["target"].value_counts())

print(
    "\nDeclining rate:",
    round(feature_df["target"].mean(), 4)
)

Feature dataframe shape: (30000, 20)

Target distribution:
target
1    16262
0    13738
Name: count, dtype: int64

Declining rate: 0.5421


In [60]:
print("Missing values:")
print(feature_df.isna().sum())

Missing values:
impressions_90d           0
clicks_90d                0
pageviews_90d             0
sessions_90d              0
users_90d                 0
engaged_sessions_90d      0
ai_sessions_90d           0
scroll_events_90d         0
days_with_impressions     0
days_with_sessions        0
content_age_days          0
days_since_last_update    0
word_count                0
char_count                0
ctr                       0
avg_position              0
engagement_rate           0
scroll_rate               0
ai_traffic_pct            0
target                    0
dtype: int64


## 3. Train + compare vs my baseline

I trained a Logistic Regression model using the same 30,000-page development population used for the Week-4 rule-based baseline.

The starter content-level dataset does not expose an observation date, so a true chronological train/test split cannot be reconstructed from this table alone. I therefore use a stratified 80/20 holdout for development evaluation.

The model uses 19 numeric features describing visibility, traffic, engagement, content age, freshness, and content size. The target is the decline label derived from the development dataset.

The Logistic Regression model achieved a ROC-AUC of 0.6777 and Average Precision of 0.6945 on the 6,000-page test set.

The baseline is evaluated on the same test pages using the same metrics. This makes the comparison a development-level decision-support comparison rather than a claim of future production performance.


In [61]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

# -----------------------------
# Prepare X and y
# -----------------------------

X = feature_df.drop(columns=["target"]).copy()
y = feature_df["target"].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

# -----------------------------
# Stratified holdout split
# -----------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining rows:", len(X_train))
print("Test rows:", len(X_test))

print("\nTraining decline rate:", round(y_train.mean(), 4))
print("Test decline rate:", round(y_test.mean(), 4))

# -----------------------------
# Logistic Regression
# -----------------------------

model = Pipeline([
    ("scaler", StandardScaler()),
    ("logistic", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

model.fit(X_train, y_train)

# Probability of decline
ml_scores = model.predict_proba(X_test)[:, 1]

# -----------------------------
# Model metrics
# -----------------------------

roc_auc = roc_auc_score(y_test, ml_scores)
pr_auc = average_precision_score(y_test, ml_scores)

print("\nLOGISTIC REGRESSION RESULTS")
print("---------------------------")
print("ROC-AUC:", round(roc_auc, 4))
print("Average Precision:", round(pr_auc, 4))

X shape: (30000, 19)
y shape: (30000,)

Training rows: 24000
Test rows: 6000

Training decline rate: 0.5421
Test decline rate: 0.542

LOGISTIC REGRESSION RESULTS
---------------------------
ROC-AUC: 0.6777
Average Precision: 0.6945


In [64]:
baseline_auc = roc_auc_score(
    y_test,
    baseline_scores
)

baseline_ap = average_precision_score(
    y_test,
    baseline_scores
)

print("MODEL COMPARISON")
print("----------------")
print("Baseline ROC-AUC:", round(baseline_auc, 4))
print("ML ROC-AUC:", round(roc_auc, 4))

print()

print("Baseline Average Precision:", round(baseline_ap, 4))
print("ML Average Precision:", round(pr_auc, 4))

MODEL COMPARISON
----------------
Baseline ROC-AUC: 0.5787
ML ROC-AUC: 0.6777

Baseline Average Precision: 0.5699
ML Average Precision: 0.6945


In [65]:
auc_difference = roc_auc - baseline_auc
ap_difference = pr_auc - baseline_ap

print("MODEL IMPROVEMENT")
print("-----------------")
print("ROC-AUC difference:", round(auc_difference, 4))
print("Average Precision difference:", round(ap_difference, 4))

if auc_difference > 0:
    print("ML has higher ROC-AUC than the baseline.")
elif auc_difference < 0:
    print("Baseline has higher ROC-AUC than ML.")
else:
    print("Both models have the same ROC-AUC.")

if ap_difference > 0:
    print("ML has higher Average Precision than the baseline.")
elif ap_difference < 0:
    print("Baseline has higher Average Precision than ML.")
else:
    print("Both models have the same Average Precision.")

MODEL IMPROVEMENT
-----------------
ROC-AUC difference: 0.099
Average Precision difference: 0.1246
ML has higher ROC-AUC than the baseline.
ML has higher Average Precision than the baseline.


## 4. Errors and interpretation

The Logistic Regression model achieved a ROC-AUC of 0.6777 and Average Precision of 0.6945 on the held-out development set.

The model's errors are useful because a probability score does not always correspond to an actual decline. False positives represent pages that the model considered at risk but were not labeled as declining, while false negatives represent declining pages that received a lower risk score.

The model mainly relies on the available traffic, visibility, engagement, freshness, and content-size signals. The coefficient ranking is used as a directional interpretation of which features influence the model most strongly.

I treat these findings as decision-support signals rather than causal explanations. A high model score means the page resembles pages associated with decline in the development data; it does not prove that a particular feature caused the decline.

The comparison with the rule-based baseline is also treated as directional. The purpose is to test whether a simple ML ranking adds useful signal beyond the transparent baseline, not to claim production-ready performance.


In [66]:
# ============================================
# SECTION 4 — ERROR ANALYSIS
# ============================================

comparison = X_test.copy()

comparison["actual"] = y_test.values
comparison["ml_score"] = ml_scores
comparison["baseline_score"] = baseline_scores

# Model predictions using 0.5 probability threshold
comparison["ml_prediction"] = (
    comparison["ml_score"] >= 0.5
).astype(int)

# False positives
false_positives = comparison[
    (comparison["ml_prediction"] == 1) &
    (comparison["actual"] == 0)
]

# False negatives
false_negatives = comparison[
    (comparison["ml_prediction"] == 0) &
    (comparison["actual"] == 1)
]

print("ERROR ANALYSIS")
print("--------------")
print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

print("\nMost important Logistic Regression coefficients:")

coefficients = pd.Series(
    model.named_steps["logistic"].coef_[0],
    index=X.columns
).sort_values(
    key=abs,
    ascending=False
)

print(coefficients.head(10))

ERROR ANALYSIS
--------------
False positives: 1226
False negatives: 935

Most important Logistic Regression coefficients:
users_90d                -1.251989
sessions_90d              1.113328
days_with_impressions     0.619343
word_count                0.461726
days_with_sessions       -0.454470
content_age_days         -0.328461
char_count               -0.307405
scroll_events_90d         0.239142
days_since_last_update    0.154724
avg_position             -0.130880
dtype: float64


The Logistic Regression model produced 1,226 false positives and 935 false negatives at the 0.5 probability threshold.

The largest absolute coefficients were observed for `users_90d`, `sessions_90d`, `days_with_impressions`, `word_count`, and `days_with_sessions`. These coefficients show which features the fitted model relies on most strongly, but they should be interpreted directionally rather than causally.

The errors show that the model does not perfectly separate declining and non-declining pages. Some pages predicted as declining did not receive the decline label, while some actual declining pages received lower predicted probabilities.

The model's ROC-AUC of 0.6777 is higher than the baseline ROC-AUC of 0.5787, while Average Precision is also higher at 0.6945 versus 0.5699. The measured differences are +0.0990 ROC-AUC and +0.1246 Average Precision.

These results suggest that the Logistic Regression model provides additional ranking signal over the rule-based baseline on this development holdout. However, because the dataset does not provide an observation date for this content-level table, these results should be treated as development-level evidence rather than proof of future production performance.


## Self-check

Before submission, I confirm:

* ✅ Every section above is filled — markdown thinking and the code that backs it.
* ✅ The notebook runs top to bottom with no errors using **Runtime → Run all**.
* ✅ No client names, URLs, or private queries are included.
* ✅ My claims use careful words such as **observed, measured, directional, and decision-support**.
* ✅ The notebook is committed to my GitHub repository under `work/notebooks/`.
* ✅ The Logistic Regression model and rule-based baseline are evaluated on the same held-out test set.
* ✅ The reported results are presented as development-level evidence, not as production or future-month performance.
